# Modelo TF-IDF Avanzado — Word + Char N-grams
**Resultado:** Val Macro ROC-AUC = **0.9034** (vs 0.8993 del primer modelo del compañero, +0.0041)

**Mejoras sobre primer_modelo:**
1. Char TF-IDF `(3,5)` 50k features en paralelo con word TF-IDF `(1,2)` 50k → 100k features totales
2. Title duplicado en el texto (mayor peso semántico del título para detectar género)
3. Lematización (reduce variantes morfológicas: *fighting/fought → fight*)
4. C=1.0 óptimo encontrado por ablación (vs C=4 del primer modelo)

**No requiere GPU.** Tiempo ~5 min en CPU.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import re, ast
import numpy as np
import pandas as pd
from scipy.sparse import hstack

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

for res in ('stopwords', 'wordnet', 'omw-1.4'):
    nltk.download(res, quiet=True)

STOP = set(stopwords.words('english'))
lem  = WordNetLemmatizer()


## 1. Carga de datos

In [ ]:
train = pd.read_csv(
    'https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTraining.zip',
    encoding='UTF-8', index_col=0
)
test = pd.read_csv(
    'https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTesting.zip',
    encoding='UTF-8', index_col=0
)
print(f"Train: {train.shape}  |  Test: {test.shape}")


## 2. Preprocesamiento

In [ ]:
def clean(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return ' '.join(
        lem.lemmatize(t) for t in text.split()
        if t not in STOP and len(t) > 2
    )

# Title se repite dos veces → mayor peso en el vector final
train['text'] = (train['title'].fillna('') + ' ' +
                 train['title'].fillna('') + ' ' +
                 train['plot'].fillna('')).apply(clean)

test['text']  = (test['title'].fillna('') + ' ' +
                 test['title'].fillna('') + ' ' +
                 test['plot'].fillna('')).apply(clean)

print("Ejemplo:", train['text'].iloc[0][:150])


## 3. Codificación de géneros

In [ ]:
train['genres'] = train['genres'].apply(ast.literal_eval)
mlb = MultiLabelBinarizer()
y   = mlb.fit_transform(train['genres'])
print(f"Clases ({len(mlb.classes_)}): {list(mlb.classes_)}")


## 4. Vectorización: Word TF-IDF + Char TF-IDF

| Vectorizador | analyzer | ngram_range | max_features | Captura |
|---|---|---|---|---|
| `word_vec` | word | (1,2) | 50 000 | semántica léxica, frases clave |
| `char_vec` | char_wb | (3,5) | 50 000 | morfología, nombres propios, raíces |

`char_wb` respeta límites de palabra (no cruza espacios).  
Concatenar ambas matrices sparse → **100 000 features** sin densificación.

> **Ablación realizada:** char(3,5) + word(1,2) con max_features=50k cada uno  
> y C=1.0 da el mejor resultado: **AUC = 0.9034**


In [ ]:
# ─── Vectorizadores para validación ────────────────────────────────────────
word_vec = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    max_features=50_000,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)
char_vec = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=50_000,
    min_df=3,
    max_df=0.95,
    sublinear_tf=True
)

# Split ANTES del fit (evitar data leakage)
idx_all = np.arange(len(train))
idx_tr, idx_val = train_test_split(idx_all, test_size=0.20, random_state=42)

text_tr  = train['text'].iloc[idx_tr]
text_val = train['text'].iloc[idx_val]
y_tr     = y[idx_tr]
y_val    = y[idx_val]

X_w_tr = word_vec.fit_transform(text_tr);   X_w_val = word_vec.transform(text_val)
X_c_tr = char_vec.fit_transform(text_tr);   X_c_val = char_vec.transform(text_val)

# .tocsr().copy() → necesario para compatibilidad con joblib en scikit-learn >= 1.4
X_tr  = hstack([X_w_tr, X_c_tr]).tocsr().copy()
X_val = hstack([X_w_val, X_c_val]).tocsr().copy()

print(f"Train features: {X_tr.shape}")
print(f"Val   features: {X_val.shape}")


## 5. Entrenamiento y evaluación

In [ ]:
clf = OneVsRestClassifier(
    LogisticRegression(C=1.0, max_iter=2000, solver='liblinear'),
    n_jobs=1   # n_jobs=1 evita problemas de memoria con matrices sparse en multiprocessing
)
clf.fit(X_tr, y_tr)

val_probs = clf.predict_proba(X_val)
auc_val   = roc_auc_score(y_val, val_probs, average='macro')

print(f"Val Macro ROC-AUC : {auc_val:.4f}")
print(f"Baseline (primer_modelo): 0.8993")
print(f"Mejora              : +{auc_val - 0.8993:.4f}")


## 6. Análisis por género

In [ ]:
print(f"{'Género':<15}  {'AUC':>6}  {'N_pos':>6}")
print("-" * 35)
for i, g in enumerate(mlb.classes_):
    if len(np.unique(y_val[:, i])) > 1:
        a = roc_auc_score(y_val[:, i], val_probs[:, i])
        n = int(y[:, i].sum())
        print(f"{g:<15}  {a:.4f}  {n:>6}")


## 7. Re-entrenamiento en dataset completo + predicción final

In [ ]:
# Vectorizadores frescos ajustados en TODOS los datos de entrenamiento
word_vec2 = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=50_000,
                             min_df=2, max_df=0.95, sublinear_tf=True)
char_vec2 = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=50_000,
                             min_df=3, max_df=0.95, sublinear_tf=True)

X_all_w = word_vec2.fit_transform(train['text'])
X_all_c = char_vec2.fit_transform(train['text'])
X_all   = hstack([X_all_w, X_all_c]).tocsr().copy()

# Test: .transform() con vocabulario de entrenamiento completo
X_te_w = word_vec2.transform(test['text'])
X_te_c = char_vec2.transform(test['text'])
X_te   = hstack([X_te_w, X_te_c]).tocsr().copy()

assert X_all.shape[1] == X_te.shape[1], "Dimensión incoherente"
print(f"Train full: {X_all.shape}  |  Test: {X_te.shape}")

clf_final = OneVsRestClassifier(
    LogisticRegression(C=1.0, max_iter=2000, solver='liblinear'), n_jobs=1
)
clf_final.fit(X_all, y)
y_pred_test = clf_final.predict_proba(X_te)


In [ ]:
cols = ['p_Action','p_Adventure','p_Animation','p_Biography','p_Comedy',
        'p_Crime','p_Documentary','p_Drama','p_Family','p_Fantasy',
        'p_Film-Noir','p_History','p_Horror','p_Music','p_Musical',
        'p_Mystery','p_News','p_Romance','p_Sci-Fi','p_Short',
        'p_Sport','p_Thriller','p_War','p_Western']

res = pd.DataFrame(y_pred_test, index=test.index, columns=cols)
res.to_csv('pred_tfidf_avanzado.csv', index_label='ID')

# Guardar predicciones de validación para el notebook de ensamble
np.save('val_preds_tfidf_avanzado.npy', val_probs)
np.save('y_val_shared.npy', y_val)

print(f"Submission: pred_tfidf_avanzado.csv")
print(f"Val preds:  val_preds_tfidf_avanzado.npy  (para ensamble)")
res.head()
